# Section 7: Data Integrity & Metamodel (Q61–Q68)
Ontology-driven validation, status distributions, and cross-entity consolidation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import get_session, run_sql
conn, ontology = get_session()

## Q61
List all production lines at the Columbus plant (PLANT-OH) that are currently active and available for scheduling. I only want lines that are actually operational — skip anything that's been decommissioned.

In [ ]:
# Ontology declares vg:sql_filter: "is_active = true" on ProductionLineAtPlant
print(f"Ontology filter: {ontology.get_role_filter('ProductionLineAtPlant')}")

run_sql(conn, """
    SELECT pl.line_code, pl.name, pl.line_type, pl.capacity_units_per_hour, pl.is_active
    FROM production_lines pl
    JOIN plants p ON pl.plant_id = p.id
    WHERE p.plant_code = 'PLANT-OH'
      AND pl.is_active = true
    ORDER BY pl.line_code
""")

## Q62
Pull the complete commercial terms for every supplier-ingredient relationship: unit cost, lead time in days, and minimum order quantity. We're rebuilding the sourcing matrix and I need it all. Cross-reference with invoice variances — flag any where the invoiced price differs from the catalog.

In [ ]:
run_sql(conn, """
    SELECT s.supplier_code, s.name as supplier_name,
           i.ingredient_code, i.name as ingredient_name,
           si.unit_cost as catalog_cost,
           si.lead_time_days,
           si.min_order_qty,
           iv.variance_type,
           iv.variance_amount
    FROM supplier_ingredients si
    JOIN suppliers s ON si.supplier_id = s.id
    JOIN ingredients i ON si.ingredient_id = i.id
    LEFT JOIN ap_invoices api ON api.supplier_id = s.id
    LEFT JOIN invoice_variances iv ON iv.invoice_id = api.id
    WHERE iv.variance_type = 'price' OR iv.id IS NULL
    ORDER BY s.supplier_code, i.ingredient_code
""")

## Q63
Verify that every batch in the system produced exactly one product — no batch should be linked to two or more SKUs or bulk intermediates. Is our one-product-per-batch rule holding, or do we have data integrity issues?

In [ ]:
violations = run_sql(conn, """
    SELECT batch_number, product_id, product_type, COUNT(*) as count
    FROM batches
    GROUP BY batch_number, product_id, product_type
    HAVING COUNT(*) > 1
""")
if len(violations) == 0:
    print("One-product-per-batch rule is holding. No violations found.")
    # Also verify distinct product_id count per batch
    check = run_sql(conn, """
        SELECT batch_number, COUNT(DISTINCT product_id) as product_count
        FROM batches
        GROUP BY batch_number
        HAVING COUNT(DISTINCT product_id) > 1
    """)
    print(f"Batches with multiple products: {len(check)}")
else:
    print("VIOLATIONS FOUND:")
    display(violations)

## Q64
Give me the distribution of orders across lifecycle states: how many are pending, allocated, shipped, and delivered? I want to see if we have a bottleneck in the pipeline.

In [ ]:
# Use ontology state machine metadata
sm = ontology.get_class_state_machine('Order')
print(f"Order state machine: {sm}")

run_sql(conn, """
    SELECT status, COUNT(*) as order_count,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) as pct
    FROM orders
    GROUP BY status
    ORDER BY CASE status
        WHEN 'pending' THEN 1
        WHEN 'allocated' THEN 2
        WHEN 'shipped' THEN 3
        WHEN 'delivered' THEN 4
    END
""")

## Q65
Pull the specific order line for order ORD-1-CLUB-DC-001-1, line number 3. Show the SKU, quantity, price, and status for just that one line.

In [ ]:
run_sql(conn, """
    SELECT o.order_number, ol.line_number,
           s.sku_code, s.name as sku_name,
           ol.quantity_cases, ol.unit_price, ol.status
    FROM orders o
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    WHERE o.order_number = 'ORD-1-CLUB-DC-001-1'
      AND ol.line_number = 3
""")

## Q66
What was our distribution center inventory snapshot on day 180? For every DC, show each SKU on hand and the quantity in cases. I need a point-in-time picture — don't mix in plant or retail inventory.

In [ ]:
run_sql(conn, """
    SELECT dc.dc_code, dc.type as dc_type, dc.name as dc_name,
           s.sku_code, s.name as sku_name,
           inv.quantity_cases
    FROM inventory inv
    JOIN distribution_centers dc ON inv.location_id = dc.id
    JOIN skus s ON inv.sku_id = s.id
    WHERE inv.day = 180
      AND inv.location_type IN ('rdc', 'customer_dc')
      AND inv.quantity_cases > 0
    ORDER BY dc.dc_code, s.sku_code
""")

## Q67
Run a sanity check on our SKU supersession data: are there any cycles in the rename chains? A SKU that eventually loops back to itself would corrupt our alias resolution.

In [ ]:
df = run_sql(conn, """
    WITH RECURSIVE chain AS (
        SELECT id, sku_code, supersedes_sku_id, ARRAY[id] as path, false as is_cycle
        FROM skus
        WHERE supersedes_sku_id IS NOT NULL

        UNION ALL

        SELECT s.id, s.sku_code, s.supersedes_sku_id,
               c.path || s.id,
               s.id = ANY(c.path)
        FROM chain c
        JOIN skus s ON c.supersedes_sku_id = s.id
        WHERE NOT s.id = ANY(c.path)
          AND s.supersedes_sku_id IS NOT NULL
          AND array_length(c.path, 1) < 20
    )
    SELECT * FROM chain WHERE is_cycle = true
""")
if len(df) == 0:
    print("No cycles found in SKU supersession chains. Data is clean.")
else:
    print("WARNING: Cycles detected!")
    display(df)

## Q68
Give me a consolidated list of every transaction document in the system — purchase orders, goods receipts, orders, shipments, returns, AP invoices, and AR invoices — with their document number and current status. I want a single view across all document types.

In [ ]:
run_sql(conn, """
    SELECT 'PurchaseOrder' as doc_type, po_number as doc_number, status FROM purchase_orders
    UNION ALL
    SELECT 'GoodsReceipt', gr_number, status FROM goods_receipts
    UNION ALL
    SELECT 'Order', order_number, status FROM orders
    UNION ALL
    SELECT 'Shipment', shipment_number, status FROM shipments
    UNION ALL
    SELECT 'Return', return_number, status FROM returns
    UNION ALL
    SELECT 'APInvoice', invoice_number, status FROM ap_invoices
    UNION ALL
    SELECT 'ARInvoice', invoice_number, status FROM ar_invoices
    ORDER BY doc_type, doc_number
""")

In [ ]:
conn.close()
print("Session closed.")